In [1]:
# 26.08.2026 - JEDEN dedykowany watek: SKAN (nie wykresy - podzial jak
# 20260819a/20260820a.ipynb i 20260822c/d.ipynb, zeby przypadkowe ponowne
# odpalenie "od gory" nie wywolalo wielogodzinnego skanu na nowo).
#
# Cel: przygotowanie danych pod globalna mape efektu wg polozenia stacji
# (prosba Homoli #6, 24.08 - patrz project_homola_review_feedback.md w
# pamieci Claude'a). Maciek zazadal wprost (26.08): wartosc per stacja na
# mapie MUSI byc policzona "dokladnie ta metodologia", ktora zamknela
# watek t0-discrepancy 24.08 (patrz project_cycle_to_cycle_map.md) -
# CZYLI NIE nasza zwykla konwencja dt=0/krok 24h (uzywana we wszystkich
# `cycle_scan_*.csv` z 19-24.08 i w calej dzisiejszej analizie jakosci
# danych), tylko dokladny przepis artykulu zweryfikowany w
# `20260824f.ipynb` dla samej Moskwy:
#   - Δt=15 dni (lag z sekcji 4/Fig.2 artykulu)
#   - krok 6h (podpis Fig.2: "20 malych przesuniec w 5 dniach" = 5d/20=6h)
#   - P_days=3350, d_days=5, m=4.0 (ta sama konfiguracja co zweryfikowany
#     test Moskwy)
#
# ROZNICA wzgledem 20260824f.ipynb: tam skan byl CELOWO ograniczony do
# okna 2004-2013 (bo szukalismy jednego konkretnego punktu z artykulu).
# TU potrzebujemy uczciwego optimum PER STACJA (domyka to najstarszy
# punkt z listy Homoli, 24.07: "t0 optymalizowane osobno per stacja, z
# lagiem") - wiec skanujemy PELNA dostepna historie kazdej stacji, nie
# jedno wskazane okno.
#
# ZAKRES STACJI: wszystkie oprocz MCRL i NANM (wykluczone decyzja z
# 26.08, patrz project_station_data_quality.md - zla jakosc danych
# potwierdzona trzema niezaleznymi sposobami, wlaczenie ich do drogiego
# skanu byloby marnowaniem czasu obliczeniowego).
#
# UWAGA O KOSZCIE (WAZNE): to REALNY, DUZY skan - ~941 000 kandydatow t0
# lacznie (suma po 20 stacjach, kazda cala wlasna historia x krok 6h),
# oszacowanie na bazie zweryfikowanego tempa z 20260824f.ipynb
# (13153 kandydatow / ~7.5 min jednowatkowo): jednowatkowo rzedu 9h,
# rownolegle (run_t0_scan_parallel, ten komputer: 20 watkow logicznych)
# realistycznie ~30-60 minut. ODPALAC W TLE (nie czekac interaktywnie) -
# stacje posortowane od najtanszej do najdrozszej (patrz komorka 3), zapis
# ATOMOWY PER STACJA (run_t0_scan_parallel zapisuje od razu po kazdej
# stacji), wiec przerwanie w polowie NIE traci juz policzonych stacji -
# mozna bezpiecznie wznowic (komorka 3 pomija stacje z istniejacym juz
# plikiem wynikowym).
import os
import sys
import time

import numpy as np
import pandas as pd
from scipy.stats import binom, norm

sys.path.insert(0, "..")
from mc_parallel import run_t0_scan_parallel

RESULTS_DIR = "../results"
MOSC_PATH = "../data/mosc_data.csv"
OULU_PATH = "../data/oulu_5min_data.csv"
EXTENDED_DIR = "../data/csv_data_stations_extended"
FULL6H_DIR = "../data/csv_data_stations_full6h"
USGS_EXTENDED_PATH = "../data/usgs_data/usgs_m4_1965_2025.csv"

P_DAYS = 3350
D_DAYS = 5
M_THRESHOLD = 4.0
DT_DAYS = 15  # przepis artykulu (Fig.2/sekcja 4) - NIE nasza zwykla konwencja dt=0

# Wszystkie stacje UZYWANE W ANALIZIE (bez MCRL, NANM - wykluczone 26.08,
# patrz project_station_data_quality.md).
OTHER_STATIONS = [
    "THUL", "LMKS", "APTY", "SOPB", "JUNG1", "SOPO", "FSMT",
    "JUNG", "NEWK", "PWNK", "MXCO", "NAIN", "HRMS", "TERA", "INVK", "ATHN",
    "AATB", "PSNM",
]


def load_mosc():
    df = pd.read_csv(MOSC_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def load_oulu():
    df = pd.read_csv(OULU_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df.dropna(subset=["datetime"])
    return df.set_index("datetime").sort_index()["value"].resample("6h").mean()


def load_station(station):
    ext_path = os.path.join(EXTENDED_DIR, f"{station.lower()}_extended_6h.csv")
    full_path = os.path.join(FULL6H_DIR, f"{station.lower()}_full_6h.csv")
    path = ext_path if os.path.exists(ext_path) else full_path
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def load_earthquakes_extended(min_mag=M_THRESHOLD):
    df = pd.read_csv(USGS_EXTENDED_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()


def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    if n_total == 0:
        return dict(N=N, N_valid=0, Np=0, Nm=0, PPDF=np.nan, PCDF=np.nan, sigma=np.nan)

    ppdf = binom.pmf(Np, n_total, 0.5)
    pcdf = binom.sf(Np - 1, n_total, 0.5)
    sigma = norm.isf(pcdf)

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma)


eq = load_earthquakes_extended(M_THRESHOLD)
print(f"EQ (rozszerzony katalog, M>={M_THRESHOLD}): {len(eq)} zdarzen, {eq.index.min()} .. {eq.index.max()}")

EQ (rozszerzony katalog, M>=4.0): 507919 zdarzen, 1965-01-01 08:04:17.780000 .. 2025-01-31 23:57:39.481000


In [2]:
# Wczytanie CR wszystkich 20 stacji (bez MCRL/NANM). Tanie (tylko odczyt
# CSV) - nie mylic z samym skanem nizej.
cr_series = {"mosc": load_mosc(), "oulu": load_oulu()}
for station in OTHER_STATIONS:
    cr_series[station.lower()] = load_station(station)

for name, s in cr_series.items():
    print(f"{name}: {len(s)} pomiarow, {s.index.min()} .. {s.index.max()}")
print(f"\nLacznie {len(cr_series)} stacji.")

mosc: 91824 pomiarow, 1960-01-01 00:00:00 .. 2025-03-23 18:00:00
oulu: 81816 pomiarow, 1970-01-01 00:00:00 .. 2025-12-31 18:00:00
thul: 96178 pomiarow, 1957-08-13 12:00:00 .. 2025-12-31 18:00:00
lmks: 60784 pomiarow, 1981-12-01 00:00:00 .. 2023-07-10 18:00:00
apty: 37234 pomiarow, 2000-07-01 00:00:00 .. 2025-12-31 18:00:00
sopb: 29044 pomiarow, 1997-12-31 00:00:00 .. 2025-12-31 18:00:00
jung1: 57538 pomiarow, 1986-01-01 00:00:00 .. 2025-12-31 18:00:00
sopo: 80347 pomiarow, 1964-03-01 00:00:00 .. 2025-12-31 18:00:00
fsmt: 36703 pomiarow, 2000-10-04 00:00:00 .. 2025-12-31 18:00:00
jung: 97003 pomiarow, 1958-10-01 06:00:00 .. 2025-12-31 18:00:00
newk: 89348 pomiarow, 1964-07-01 00:00:00 .. 2025-12-31 18:00:00
pwnk: 34251 pomiarow, 2000-09-14 00:00:00 .. 2025-12-31 18:00:00
mxco: 50411 pomiarow, 1990-01-01 00:00:00 .. 2025-10-14 12:00:00
nain: 36291 pomiarow, 2000-11-10 18:00:00 .. 2025-12-31 18:00:00
hrms: 88955 pomiarow, 1957-05-29 06:00:00 .. 2021-12-01 06:00:00
tera: 70160 pomiarow, 19

In [3]:
# GLOWNA KOMORKA - SKAN. Rzeczywisty, wielogodzinny (jednowatkowo) /
# kilkudziesieciominutowy (rownolegle) koszt obliczeniowy - patrz uwaga w
# komorce 0. NIE odpalac interaktywnie/przez przypadek - zalecane tlo
# (np. "Run All" zostawione dzialac, albo jupyter nbconvert --execute w
# tle).
#
# Kolejnosc: od najtanszej stacji (najkrotsza historia) do najdrozszej -
# szybka informacja zwrotna, ze wszystko dziala, zanim zaczna sie
# najdluzsze skany. Kazda stacja zapisywana ATOMOWO od razu po policzeniu
# (run_t0_scan_parallel), plik pomijany jesli juz istnieje (bezpieczne
# wznowienie po przerwaniu). try/except per stacja - pojedynczy blad na
# jednej stacji (np. nieoczekiwany ksztalt danych) nie ma zepsuc/przerwac
# reszty wielogodzinnego skanu.
plans = []
for station, cr in cr_series.items():
    t0_first = max(cr.index.min(), eq.index.min())
    t0_last = min(cr.index.max() - pd.Timedelta(days=P_DAYS),
                  eq.index.max() - pd.Timedelta(days=P_DAYS + DT_DAYS))
    if t0_last <= t0_first:
        print(f"{station}: pomijam - brak wspolnego zakresu CR/EQ dla P_days={P_DAYS}")
        continue
    t0_candidates = pd.date_range(t0_first.ceil("6h"), t0_last.floor("6h"), freq="6h")
    plans.append((station, t0_candidates))

plans.sort(key=lambda p: len(p[1]))  # najtansza stacja pierwsza
print("Kolejnosc skanu (stacja: liczba kandydatow):")
for station, t0c in plans:
    print(f"  {station}: {len(t0c)}")
print(f"\nSuma kandydatow: {sum(len(t0c) for _, t0c in plans)}")

overall_start = time.time()
for i, (station, t0_candidates) in enumerate(plans, 1):
    save_path = f"{RESULTS_DIR}/finetune_{station}_P{P_DAYS}_d{D_DAYS}_dt{DT_DAYS}_fullhistory.csv"
    if os.path.exists(save_path):
        print(f"[{i}/{len(plans)}] {station}: plik juz istnieje ({save_path}), pomijam")
        continue

    print(f"[{i}/{len(plans)}] {station}: start, {len(t0_candidates)} kandydatow "
          f"({t0_candidates.min().date()} .. {t0_candidates.max().date()})...")
    t_start = time.time()
    try:
        scan_end = t0_candidates.max() + pd.Timedelta(days=P_DAYS + DT_DAYS)
        cr_clipped = cr_series[station][
            (cr_series[station].index >= t0_candidates.min()) & (cr_series[station].index <= scan_end)
        ]
        eq_clipped = eq[(eq.index >= t0_candidates.min()) & (eq.index <= scan_end)]

        result = run_t0_scan_parallel(
            cr_clipped, eq_clipped, t0_candidates,
            P_days=P_DAYS, d_days=D_DAYS, m=M_THRESHOLD, dt_days=DT_DAYS,
            stat_fn=cosmoseismic_stat,
            save_path=save_path,
        )
        elapsed_min = (time.time() - t_start) / 60
        print(f"[{i}/{len(plans)}] {station}: GOTOWE - {len(result)} wierszy w {elapsed_min:.1f} min "
              f"({len(t0_candidates) / max(elapsed_min * 60, 1e-6):.1f} kandydatow/s)")
    except Exception as exc:
        print(f"[{i}/{len(plans)}] {station}: BLAD - {exc!r} - POMIJAM te stacje, kontynuuje reszte")

total_elapsed_min = (time.time() - overall_start) / 60
print(f"\nCALOSC: {total_elapsed_min:.1f} min ({total_elapsed_min / 60:.2f} h)")

Kolejnosc skanu (stacja: liczba kandydatow):
  psnm: 11596
  nain: 21933
  athn: 21933
  fsmt: 22084
  pwnk: 22164
  apty: 22464
  invk: 23192
  sopb: 26116
  mxco: 37800
  jung1: 43644
  lmks: 47388
  aatb: 62636
  oulu: 67020
  tera: 69460
  hrms: 69752
  mosc: 74322
  thul: 74322
  sopo: 74322
  jung: 74322
  newk: 74322

Suma kandydatow: 940792
[1/20] psnm: start, 11596 kandydatow (2007-12-09 .. 2015-11-15)...
[1/20] psnm: GOTOWE - 11596 wierszy w 0.8 min (239.4 kandydatow/s)
[2/20] nain: start, 21933 kandydatow (2000-11-10 .. 2015-11-15)...
[2/20] nain: GOTOWE - 21933 wierszy w 1.6 min (233.4 kandydatow/s)
[3/20] athn: start, 21933 kandydatow (2000-11-10 .. 2015-11-15)...
[3/20] athn: GOTOWE - 21933 wierszy w 1.6 min (234.4 kandydatow/s)
[4/20] fsmt: start, 22084 kandydatow (2000-10-04 .. 2015-11-15)...
[4/20] fsmt: GOTOWE - 22084 wierszy w 1.6 min (233.5 kandydatow/s)
[5/20] pwnk: start, 22164 kandydatow (2000-09-14 .. 2015-11-15)...
[5/20] pwnk: GOTOWE - 22164 wierszy w 1.6 min 

In [4]:
# KONTROLA SPOJNOSCI (tanie, do odpalenia po zakonczeniu skanu Moskwy):
# ten pelnohistoryczny skan Moskwy POWINIEN znalezc co najmniej tak dobre
# minimum jak zawezony test z 20260824f.ipynb (okno 2004-2013 bylo
# PODZBIOREM pelnej historii) - t0=2008-09-27, PPDF~1.6e-9. Jesli globalne
# minimum pelnej historii jest GORSZE od tego punktu, to sygnal bledu w
# tym notebooku (np. zla klipsacja/przesuniecie edges), nie tylko "inny
# wynik" - do sprawdzenia PRZED zbudowaniem mapy na tych danych.
KNOWN_MOSC_T0 = pd.Timestamp("2008-09-27")
KNOWN_MOSC_PPDF = 1.6e-9

mosc_path = f"{RESULTS_DIR}/finetune_mosc_P{P_DAYS}_d{D_DAYS}_dt{DT_DAYS}_fullhistory.csv"
if os.path.exists(mosc_path):
    mosc_full = pd.read_csv(mosc_path, parse_dates=["t0"]).dropna(subset=["PPDF"])
    mosc_full = mosc_full[mosc_full["PPDF"] > 0]
    best = mosc_full.loc[mosc_full["PPDF"].idxmin()]
    print(f"Globalne minimum Moskwy (pelna historia): t0={best['t0']}, PPDF={best['PPDF']:.3e}")
    print(f"Znany wynik z 20260824f.ipynb (okno 2004-2013): t0={KNOWN_MOSC_T0.date()}, PPDF={KNOWN_MOSC_PPDF:.3e}")
    if best["PPDF"] <= KNOWN_MOSC_PPDF * 1.05:  # tolerancja na drobne roznice numeryczne
        print("OK: pelnohistoryczne minimum jest co najmniej tak dobre jak znany wczesniejszy wynik.")
    else:
        print("UWAGA: pelnohistoryczne minimum jest WYRAZNIE SLABSZE niz znany wczesniejszy wynik "
              "z tego samego okna czasowego - sprawdzic notebook przed uzyciem wynikow do mapy.")
else:
    print(f"Brak jeszcze pliku {mosc_path} - odpal najpierw glowna komorke skanu (mosc jest w kolejce "
          "wsrod ostatnich, najdrozszych stacji).")

Globalne minimum Moskwy (pelna historia): t0=1968-07-12 18:00:00, PPDF=1.904e-10
Znany wynik z 20260824f.ipynb (okno 2004-2013): t0=2008-09-27, PPDF=1.600e-09
OK: pelnohistoryczne minimum jest co najmniej tak dobre jak znany wczesniejszy wynik.
